# Sensor Sequence Explorer

Interactive panels over the raw CMI data, built directly on the
`SignalCleaner` / `MotionFilter` / `IMUExtractor` / `RotationExtractor` /
`SequenceExtractor` pipeline in `base_utils_qwen.py`. All logic lives in
`sequence_explorer_utils.py` — this notebook only wires it up.

| Tab | What it does |
|---|---|
| **Catalogue** | Filter by subject / gesture / orientation / type / behavior / phase. `+ Add current` or `+ Add all filtered` builds up a working list of `sequence_id`s one selection at a time — pick an orientation and add, then pick a different subject and add again. The count and a preview update live, and a small-multiples grid shows whatever's currently catalogued. |
| **Features** | Before vs after on one sequence, run through the *actual* `SequenceExtractor` — every constructor parameter is exposed (modes, motion filter, kalman noise, dead reckoning, cleaning, sampling rates, resampling, chunking, frame stats). Both plots are zero-centred so offset and drift are visible at a glance. |
| **Spectrum** | FFT / Welch PSD / spectrogram of the **raw → velocity → displacement → jerk** chain for one IMU axis, computed via the literal `SignalCleaner → MotionFilter → IMUExtractor` calls `SequenceExtractor.fit` makes internally — not a re-implementation. |
| **Clusters** | Frame-level features → scale → PCA/t-SNE/UMAP → KMeans/GMM/Agglomerative/DBSCAN/HDBSCAN, over either the **catalogue** from tab 1 or the **current filter** selection. |

The parameter accordion (Modes / Motion filter / Cleaning / Sampling rates /
Chunking) is shared across Features, Spectrum and Clusters, so a setting you
dial in stays consistent everywhere you look.

In [ ]:
import os
import sys
import warnings

import pandas as pd

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
%matplotlib inline

current_dir = os.getcwd()
workspace_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == 'notebooks' else current_dir
sys.path.insert(0, workspace_root)
sys.path.insert(0, os.path.join(workspace_root, 'src'))

try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/src')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception:
    pass

try:
    from src import data_utils
    from src.sequence_explorer_utils import SequenceExplorer
except ImportError:
    import data_utils
    from sequence_explorer_utils import SequenceExplorer

print('Workspace root:', workspace_root)

## 1. Configuration

In [ ]:
use_sample_data = False          # True -> data/sample.csv, False -> data/train.csv
sample_file = 'sample.csv'
label_col = 'gesture'          # true-label panel + cluster`` purity
sampling_rate_hz = 20.0        # IMU native rate; drives every frequency axis
random_state = 42

## 2. Load

In [ ]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_df["sequence_id"].nunique()} sequences')
else:
    raw_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_df["sequence_id"].nunique()} sequences')

df = raw_df.set_index('row_id').copy(deep=True)
df['gesture'] = df['gesture'].fillna('non_bfrb').astype(str)
df['orientation'] = df['orientation'].fillna('Unknown').astype(str)
df['is_target'] = df['sequence_type'].eq('Target').astype(int)
df['bfrb'] = df['gesture'].where(df['is_target'].astype(bool), 'non_bfrb')

print(f'{df.shape[0]:,} rows | {df["sequence_id"].nunique():,} sequences | '
      f'{df["subject"].nunique()} subjects')

## 3. Explore

Building a catalogue: pick filters (e.g. `orientation`), hit **+ Add all
filtered**, change the filters (e.g. a different `subject`), hit it again —
the catalogue accumulates across selections. Switch the Clusters tab's
**Population** to `catalogue` to cluster exactly that working set.

On full `train.csv`, keep **Max seqs** in the Clusters tab at 400 or below —
the extractor reruns on every control change.

In [ ]:
explorer = SequenceExplorer(
    df,
    label_col=label_col,
    fs=sampling_rate_hz,
    random_state=random_state,
)
explorer.show()

## 4. Non-interactive calls

Every panel is a plain bound method, and `_extractor_kwargs` turns a
plain dict into the exact kwargs `SequenceExtractor` takes — useful for
scripting a parameter sweep outside the widgets.

In [ ]:
seq_id = explorer.meta_.index[0]

param_vals = dict(
    acc_modes=('raw', 'velocity', 'jerk'),
    rotation_modes=('quaternion', 'angular_velocity'),
    tof_modes=('pooled_stats',),
    thm_modes=('centered_diff',),
    motion_filter_mode='kalman',
    use_dead_reckoning=False,
    dead_reckoning_detrend=False,
    kalman_process_noise=-3,          # log10
    kalman_measurement_noise=-2,      # log10
    compute_dt=True,
    window_size=7,
    smooth_alpha=0.0,
    clip_value=0.0,
    interp_mode='linear',
    maxlen=160,
    padding_value=0.0,
    imu_native_sampling_rate=20,
    imu_target_sampling_rate=20,
    rot_native_sampling_rate=20,
    rot_target_sampling_rate=20,
    tof_native_sampling_rate=5,
    tof_target_sampling_rate=5,
    thm_native_sampling_rate=5,
    thm_target_sampling_rate=5,
    chunk_window_size=0,               # 0 -> None
    chunk_stride=0,
    frame_stats=('mean', 'std', 'min', 'max', 'last'),
    add_global_context=False,
    resample_modalities=False,
)

explorer.update_spectrum(seq_id, '', 'acc_x', 'fft', True, True, **param_vals)

In [ ]:
# Kalman process-noise sweep, reading straight off the real MotionFilter
for q in [-4, -3, -2]:
    kwargs = explorer._extractor_kwargs({**param_vals, 'motion_filter_mode': 'kalman',
                                          'kalman_process_noise': q})
    domains, dt = explorer._imu_domain_chain(explorer.sequence(seq_id), 'acc_x', kwargs)
    print(f'Q=1e{q:g}  filtered acc_x std = {domains["raw"].std():.4f}')